# 00 — Run the full NEON pipeline

This is the cleaned, documented counterpart to the repository's active `Raster_processing.ipynb`. It uses the same public `go_forth_and_multiply` orchestrator, then checks the files written for each flightline. Existing valid artifacts are reused, so the same cell can restart an interrupted run.

## 1. Setup

From a fresh clone, install the package with `python -m pip install -e ".[notebooks]"`, start Jupyter from the repository root, and select the Python 3 kernel. The imports below intentionally match the short public import used during development.

In [ ]:
from pathlib import Path

from spectralbridge import go_forth_and_multiply

RUN = False
base_folder = Path("outputs/neon_notebook")
site_code = "NIWO"
product_code = "DP1.30006.001"
year_month = "2023-08"
flight_lines = [
    "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance",
]
engine = "thread"
max_workers = 1

## 2. Run or resume

Start with one flightline and one worker. Set `RUN = True` only after checking the identifiers and available storage. A later rerun checks canonical outputs and resumes at the first missing or invalid stage.

In [ ]:
print(f"Output root: {base_folder.resolve()}")
print(f"Flightlines: {len(flight_lines)} | engine={engine} | workers={max_workers}")

if RUN:
    go_forth_and_multiply(
        base_folder=base_folder,
        site_code=site_code,
        product_code=product_code,
        year_month=year_month,
        flight_lines=flight_lines,
        engine=engine,
        max_workers=max_workers,
        extraction_mode="full",
    )
else:
    print("Dry run only. Review the configuration, then set RUN = True.")

## 3. Check the outputs

Like the active research notebook, this cell inspects the output directory rather than assuming a successful function return means every product is ready. It is safe to run before, during, or after processing.

In [ ]:
for flight_stem in flight_lines:
    flight_dir = base_folder / flight_stem
    files = sorted(path for path in flight_dir.glob("*") if path.is_file())
    print(f"\n{flight_stem}")
    print(f"  directory exists: {flight_dir.exists()}")
    print(f"  ENVI headers: {sum(path.suffix == '.hdr' for path in files)}")
    print(f"  ENVI images: {sum(path.suffix == '.img' for path in files)}")
    print(f"  Parquet tables: {sum(path.suffix == '.parquet' for path in files)}")
    print(f"  QA artifacts: {sum('_qa' in path.stem for path in files)}")
    for path in files:
        if path.suffix in {'.parquet', '.json', '.png'} and ('merged' in path.stem or '_qa' in path.stem):
            print(f"    {path.name} ({path.stat().st_size:,} bytes)")

## 4. Interpret and continue

A completed flightline should contain raw and corrected ENVI pairs, convolved sensor products, Parquet tables, and QA artifacts. Continue with notebook 04 to query the tables and notebook 05 to inspect images and diagnostics. If a category is missing, rerun the orchestrator; do not rename or manually move intermediate files.